# Route A: bacpipe BirdNET to JSONL to analysis

1. bacpipe extracts 1024-d BirdNET embeddings from audio.
2. The adapter writes a JSONL manifest.
3. bioacoustic-embedding-dynamics runs PCA, UMAP, trajectory, change-point, and HMM.

Use a T4 GPU: `colab new -s bel-gpu --gpu T4`. The first run downloads BirdNET weights.

## 1. Get this repo

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/path/to/bioacoustic-embedding-dynamics

# GITHUB = "https://github.com/YOUR_USER/bioacoustic-embedding-dynamics.git"
# !rm -rf bioacoustic-embedding-dynamics && git clone --depth 1 {GITHUB}
# %cd bioacoustic-embedding-dynamics

import os
print("cwd:", os.getcwd())

## 2. Install (bacpipe and this package)

In [ ]:
import sys

# bacpipe requires Python < 3.13 and pins TF/JAX/CUDA/numpy 1.26.
# Colab is 3.13 with TF and Torch already installed, so skip the full pin set.
!{sys.executable} -m pip install --ignore-requires-python --no-deps bacpipe
!{sys.executable} -m pip install -q umap-learn ruptures hmmlearn
!{sys.executable} -m pip install -q --no-deps -e .

import librosa
if not hasattr(librosa, "get_duration"):
    from librosa.core.audio import get_duration as _gd
    librosa.get_duration = _gd
print("ready")

## 3. Run bacpipe on audio

Default: bacpipe built-in test data. Set USE_YOUR_AUDIO = True and upload .wav files to /content/audio/.

In [ ]:
import librosa
if not hasattr(librosa, "get_duration"):
    from librosa.core.audio import get_duration as _gd
    librosa.get_duration = _gd

import bacpipe
from pathlib import Path

USE_YOUR_AUDIO = False
AUDIO_DIR = "/content/audio"
MODEL = "birdnet"
DEVICE = "cuda"  # T4 via `colab new -s bel-gpu --gpu T4`; fallback: "cpu"

if USE_YOUR_AUDIO:
    audio_dir = Path(AUDIO_DIR)
else:
    audio_dir = Path(bacpipe.__file__).parent / "tests" / "test_data"

bacpipe.config.models = [MODEL]
bacpipe.config.dashboard = False
bacpipe.config.audio_dir = str(audio_dir)
bacpipe.settings.device = DEVICE

print("audio_dir:", audio_dir)
bacpipe.ensure_models_exist(model_names=[MODEL])
loader = bacpipe.generate_embeddings(
    model_name=MODEL,
    audio_dir=str(audio_dir),
    check_if_already_processed=True,
)
print("embedding_size:", loader.metadata_dict.get("embedding_size"))

## 4. Convert bacpipe output to JSONL

In [ ]:
from pathlib import Path

from bioacoustic_embedding_dynamics.adapters import bacpipe_loader_to_manifest

MANIFEST = Path("data/bacpipe_birdnet.jsonl")
bacpipe_loader_to_manifest(
    {MODEL: loader},
    MODEL,
    MANIFEST,
    min_confidence=0.0,
)
print(f"Wrote {MANIFEST} ({sum(1 for _ in MANIFEST.open())} lines)")

## 5. Run embedding analysis

In [ ]:
!python -m bioacoustic_embedding_dynamics.cli --manifest {MANIFEST} --out reports/bacpipe --seed 42

## 6. Summary and figures

In [ ]:
import json
from IPython.display import Image, display

summary = json.loads(Path("reports/bacpipe/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png",
]:
    p = Path("reports/bacpipe") / name
    if p.is_file():
        display(Image(filename=str(p)))